# 10. 함수 라이브러리의 입력과 출력


## Goal

정의와 호출을 분리하고, stdin → stdout 데이터 변환 함수의 출력과 종료 상태를 확인한다. [교안 10-1](../../10-program-architecture/10-1-modules-contracts.md)과 연결된다.


## Setup

실습 폴더 안에 라이브러리를 만든다. source는 현재 Bash 셀에서만 유지되므로 사용하는 셀마다 로드한다.


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-10-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"격리된 실습 디렉터리: {lab_dir}")


## Steps

### 1. 부수 효과 없는 라이브러리 작성


In [ ]:
%%bash
set -euo pipefail
cat > "$BASH_LAB_DIR/labels.sh" <<'BASH'
classify() {
    (( $# == 1 )) || return 2
    case $1 in
        INFO) printf 'normal\n' ;;
        ERROR) printf 'review\n' ;;
        *) printf 'unsupported level\n' >&2; return 2 ;;
    esac
}
BASH
bash -n "$BASH_LAB_DIR/labels.sh"
observed=$(source "$BASH_LAB_DIR/labels.sh")
[[ -z $observed ]]
printf 'source emitted no data\n'


### 2. 함수의 출력과 종료 상태 검사


In [ ]:
%%bash
set -euo pipefail
source "$BASH_LAB_DIR/labels.sh"
[[ $(classify INFO) == normal ]]
[[ $(classify ERROR) == review ]]
status=0
classify DEBUG > "$BASH_LAB_DIR/out" 2> "$BASH_LAB_DIR/err" || status=$?
[[ $status == 2 && ! -s $BASH_LAB_DIR/out && -s $BASH_LAB_DIR/err ]]
printf 'INFO=normal ERROR=review DEBUG=status2\n'


### 3. 동적 스코프 관찰

예상: 내부 호출은 caller의 local 값을 읽는다. 재사용 함수는 값을 인수로 전달하는 방식으로 고쳐 본다.


In [ ]:
%%bash
set -euo pipefail
show_label() { printf '%s\n' "$label"; }
caller() { local label=local_value; show_label; }
label=global_value
[[ $(caller) == local_value ]]
[[ $label == global_value ]]
printf 'caller=local_value parent=global_value\n'


## Checks

- source만 했을 때 함수 호출 결과가 출력되지 않는가?
- 정상 데이터와 오류 진단이 분리되는가?
- local 값이 내부 호출에서 보이는 이유를 Python 스코프와 비교할 수 있는가?


## Next Steps

함수의 입력·출력 형식을 유지하면서 독립 작업을 제한된 동시성으로 실행한다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
